# Data Scientist Model Training

This notebook contains the Data Scientist part for the Agriculture Health AI Project. The objective is to train and compare CNN transfer learning models for plant leaf disease classification.

The models used are:
- MobileNetV3Small
- ResNet50
- DenseNet121

The models are evaluated using:
- Test accuracy
- Mean Average Precision, mAP
- Training time
- Total parameters

## 1. Dataset Extraction

This section extracts the train, validation, and test dataset ZIP files into the dataset folder.

In [1]:
from pathlib import Path
import zipfile

# Current folder should be your Project folder
project_dir = Path.cwd()
dataset_dir = project_dir / "dataset"

dataset_dir.mkdir(exist_ok=True)

# Find all ZIP files in Project folder
zip_files = list(project_dir.glob("*.zip"))

print("ZIP files found:")
for file in zip_files:
    print("-", file.name)

# Extract all ZIP files into dataset folder
for zip_path in zip_files:
    print(f"Extracting {zip_path.name}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(dataset_dir)

print("\nDone extracting.")
print("Dataset folder created at:", dataset_dir)

ZIP files found:
- test-20260605T112644Z-3-001.zip
- train-20260605T112647Z-3-001.zip
- val-20260605T112650Z-3-001.zip
Extracting test-20260605T112644Z-3-001.zip...
Extracting train-20260605T112647Z-3-001.zip...
Extracting val-20260605T112650Z-3-001.zip...

Done extracting.
Dataset folder created at: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\dataset


## 2. Dataset Summary

This section checks the number of images in each dataset split and class folder. The dataset contains 5 plant disease classes:
- leaf_blight
- leaf_rust
- leaf_spot
- powdery_mildew
- yellow_leaf_disease

In [2]:
from pathlib import Path

dataset_dir = Path.cwd() / "dataset"

for split in ["train", "val", "test"]:
    split_dir = dataset_dir / split
    print(f"\n{split.upper()} SET")
    
    total_images = 0
    
    for class_folder in sorted(split_dir.iterdir()):
        if class_folder.is_dir():
            image_count = len(list(class_folder.glob("*.jpg")))
            total_images += image_count
            print(f"{class_folder.name}: {image_count} images")
    
    print(f"Total {split}: {total_images} images")


TRAIN SET
leaf_blight: 812 images
leaf_rust: 812 images
leaf_spot: 812 images
powdery_mildew: 812 images
yellow_leaf_disease: 812 images
Total train: 4060 images

VAL SET
leaf_blight: 164 images
leaf_rust: 164 images
leaf_spot: 164 images
powdery_mildew: 164 images
yellow_leaf_disease: 164 images
Total val: 820 images

TEST SET
leaf_blight: 174 images
leaf_rust: 174 images
leaf_spot: 174 images
powdery_mildew: 174 images
yellow_leaf_disease: 174 images
Total test: 870 images


## 3. Environment Setup

This section checks the Python environment and installs TensorFlow. TensorFlow/Keras is used to build and train the CNN models.

In [4]:
import sys

print("Python version:")
print(sys.version)

print("\nPython location:")
print(sys.executable)

Python version:
3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]

Python location:
C:\Users\ASUS\anaconda3\envs\ISB46703\python.exe


In [5]:
import sys

!{sys.executable} -m pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 12.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1


In [7]:
import sys

!{sys.executable} -m pip install tensorflow

In [2]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: []


## 4. Dataset Loading and Preprocessing

The dataset is loaded using TensorFlow. Images are resized to 224 x 224 pixels and loaded in batches of 32. The training, validation, and testing datasets are prepared for CNN model training.

In [2]:
import tensorflow as tf
from pathlib import Path

# Dataset paths
dataset_dir = Path.cwd() / "dataset"

train_dir = dataset_dir / "train"
val_dir = dataset_dir / "val"
test_dir = dataset_dir / "test"

# Settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# Load training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

# Load validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

# Load testing dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

# Class names
class_names = train_ds.class_names
num_classes = len(class_names)

print("Class names:", class_names)
print("Number of classes:", num_classes)

# Performance setup
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Dataset loaded and prepared successfully.")

Found 4060 files belonging to 5 classes.
Found 820 files belonging to 5 classes.
Found 870 files belonging to 5 classes.
Class names: ['leaf_blight', 'leaf_rust', 'leaf_spot', 'powdery_mildew', 'yellow_leaf_disease']
Number of classes: 5
Dataset loaded and prepared successfully.


In [3]:
for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("First 10 labels:", labels[:10].numpy())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
First 10 labels: [2 2 2 4 4 0 4 4 0 1]


## 5. CNN Model Imports

This section imports the CNN architectures required by the project: ResNet50, DenseNet121, and MobileNetV3Small.

In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Small

print("CNN model imports successful.")

CNN model imports successful.


## 6. Output Folder Setup

This section creates folders to store model results and saved trained models.

In [7]:
from pathlib import Path

results_dir = Path.cwd() / "results"
models_dir = Path.cwd() / "saved_models"

results_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

print("Results folder:", results_dir)
print("Saved models folder:", models_dir)

Results folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results
Saved models folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\saved_models


## 7. Hyperparameter Tuning

Before final 50-epoch training, a small hyperparameter tuning experiment was conducted to identify a suitable configuration for each CNN model. Three trials were tested for each model by changing the learning rate, dropout value, and trainable layer setting.

| Trial   | Learning Rate | Dropout | Trainable Layers         |
| ------- | ------------: | ------: | ------------------------ |
| Trial 1 |         0.001 |     0.3 | Frozen base              |
| Trial 2 |        0.0001 |     0.4 | Frozen base              |
| Trial 3 |       0.00001 |     0.3 | Fine-tune last 10 layers |

Each trial was trained for 3 epochs using the training and validation datasets. The best trial was selected based on validation accuracy.

Based on the hyperparameter tuning results, Trial 1 was selected for all three models. Therefore, the final 50-epoch training used the following configuration:

* Learning rate: 0.001
* Dropout: 0.3
* Trainable layers: Frozen base
* Epochs: 50

In [6]:
import time
import gc
import pandas as pd
import tensorflow as tf
from pathlib import Path

from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Small, ResNet50, DenseNet121
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

# =========================================================
# 1. Locate project folder
# =========================================================

project_dir = Path.cwd()

# If notebook is opened inside Data_Scientist folder, go back to Project folder
if project_dir.name == "Data_Scientist":
    project_dir = project_dir.parent

dataset_dir = project_dir / "dataset"

# Save tuning results to Data_Scientist/results if available
data_scientist_results_dir = project_dir / "Data_Scientist" / "results"

if data_scientist_results_dir.exists():
    tuning_results_dir = data_scientist_results_dir
else:
    tuning_results_dir = project_dir / "results"
    tuning_results_dir.mkdir(exist_ok=True)

print("Project folder:", project_dir)
print("Dataset folder:", dataset_dir)
print("Tuning results folder:", tuning_results_dir)


# =========================================================
# 2. Make sure train_ds, val_ds, class_names, and num_classes exist
# =========================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

if "train_ds" not in globals() or "val_ds" not in globals() or "num_classes" not in globals():
    print("\nLoading dataset for hyperparameter tuning...")

    train_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_dir / "train",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        dataset_dir / "val",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED
    )

    class_names = train_ds.class_names
    num_classes = len(class_names)

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

    print("Dataset loaded successfully.")
else:
    print("\nDataset variables already exist. Using existing train_ds and val_ds.")

print("Class names:", class_names)
print("Number of classes:", num_classes)


# =========================================================
# 3. Helper function: freeze or fine-tune model layers
# =========================================================

def set_trainable_layers(base_model, fine_tune_last=0):
    """
    fine_tune_last = 0 means freeze the whole base model.
    fine_tune_last = 10 means train only the last 10 layers.
    """
    if fine_tune_last == 0:
        base_model.trainable = False
    else:
        base_model.trainable = True

        # Freeze all layers except the last few layers
        for layer in base_model.layers[:-fine_tune_last]:
            layer.trainable = False

        # Keep BatchNormalization layers frozen for stable fine-tuning
        for layer in base_model.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False


# =========================================================
# 4. Helper function: build transfer learning model
# =========================================================

def build_transfer_model(model_name, learning_rate, dropout_rate, fine_tune_last=0):
    """
    Builds a transfer learning model for:
    - MobileNetV3Small
    - ResNet50
    - DenseNet121
    """
    inputs = tf.keras.Input(shape=(224, 224, 3))

    if model_name == "MobileNetV3Small":
        base_model = MobileNetV3Small(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3),
            pooling="avg"
        )

        set_trainable_layers(base_model, fine_tune_last)
        x = base_model(inputs, training=False)

    elif model_name == "ResNet50":
        base_model = ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3),
            pooling="avg"
        )

        set_trainable_layers(base_model, fine_tune_last)
        x = layers.Lambda(resnet_preprocess)(inputs)
        x = base_model(x, training=False)

    elif model_name == "DenseNet121":
        base_model = DenseNet121(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3),
            pooling="avg"
        )

        set_trainable_layers(base_model, fine_tune_last)
        x = layers.Lambda(densenet_preprocess)(inputs)
        x = base_model(x, training=False)

    else:
        raise ValueError("Unsupported model name.")

    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name=f"{model_name}_Tuning")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# =========================================================
# 5. Hyperparameter tuning settings
# =========================================================

TUNING_EPOCHS = 3

model_names = [
    "MobileNetV3Small",
    "ResNet50",
    "DenseNet121"
]

tuning_trials = [
    {
        "Trial": "Trial 1",
        "Learning Rate": 0.001,
        "Dropout": 0.3,
        "Fine Tune Last Layers": 0,
        "Trainable Layers": "Frozen base"
    },
    {
        "Trial": "Trial 2",
        "Learning Rate": 0.0001,
        "Dropout": 0.4,
        "Fine Tune Last Layers": 0,
        "Trainable Layers": "Frozen base"
    },
    {
        "Trial": "Trial 3",
        "Learning Rate": 0.00001,
        "Dropout": 0.3,
        "Fine Tune Last Layers": 10,
        "Trainable Layers": "Fine-tune last 10 layers"
    }
]


# =========================================================
# 6. Run hyperparameter tuning
# =========================================================

tuning_results = []

for model_name in model_names:
    print("\n" + "=" * 60)
    print(f"Starting tuning for {model_name}")
    print("=" * 60)

    for trial in tuning_trials:
        print(f"\n{model_name} - {trial['Trial']}")
        print("Learning rate:", trial["Learning Rate"])
        print("Dropout:", trial["Dropout"])
        print("Trainable layers:", trial["Trainable Layers"])

        tf.keras.backend.clear_session()
        gc.collect()

        model = build_transfer_model(
            model_name=model_name,
            learning_rate=trial["Learning Rate"],
            dropout_rate=trial["Dropout"],
            fine_tune_last=trial["Fine Tune Last Layers"]
        )

        start_time = time.time()

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=TUNING_EPOCHS,
            verbose=1
        )

        end_time = time.time()
        training_time = end_time - start_time

        best_val_accuracy = max(history.history["val_accuracy"])
        final_val_accuracy = history.history["val_accuracy"][-1]
        final_val_loss = history.history["val_loss"][-1]

        tuning_results.append({
            "Model": model_name,
            "Trial": trial["Trial"],
            "Learning Rate": trial["Learning Rate"],
            "Dropout": trial["Dropout"],
            "Trainable Layers": trial["Trainable Layers"],
            "Epochs": TUNING_EPOCHS,
            "Best Validation Accuracy": best_val_accuracy,
            "Best Validation Accuracy (%)": best_val_accuracy * 100,
            "Final Validation Accuracy": final_val_accuracy,
            "Final Validation Accuracy (%)": final_val_accuracy * 100,
            "Final Validation Loss": final_val_loss,
            "Training Time (seconds)": training_time,
            "Training Time (minutes)": training_time / 60,
            "Total Parameters": model.count_params()
        })

        del model
        tf.keras.backend.clear_session()
        gc.collect()


# =========================================================
# 7. Save and display tuning results
# =========================================================

tuning_results_df = pd.DataFrame(tuning_results)

# Round values for cleaner display
tuning_results_df["Best Validation Accuracy (%)"] = tuning_results_df["Best Validation Accuracy (%)"].round(2)
tuning_results_df["Final Validation Accuracy (%)"] = tuning_results_df["Final Validation Accuracy (%)"].round(2)
tuning_results_df["Final Validation Loss"] = tuning_results_df["Final Validation Loss"].round(4)
tuning_results_df["Training Time (seconds)"] = tuning_results_df["Training Time (seconds)"].round(2)
tuning_results_df["Training Time (minutes)"] = tuning_results_df["Training Time (minutes)"].round(2)

# Save full tuning results
tuning_results_df.to_csv(
    tuning_results_dir / "hyperparameter_tuning_results.csv",
    index=False
)

# Select best trial for each model based on best validation accuracy
selected_hyperparameters_df = (
    tuning_results_df
    .sort_values(by=["Model", "Best Validation Accuracy (%)"], ascending=[True, False])
    .groupby("Model")
    .head(1)
    .reset_index(drop=True)
)

selected_hyperparameters_df = selected_hyperparameters_df[
    [
        "Model",
        "Trial",
        "Learning Rate",
        "Dropout",
        "Trainable Layers",
        "Epochs",
        "Best Validation Accuracy (%)",
        "Final Validation Accuracy (%)",
        "Training Time (minutes)"
    ]
]

# Save selected hyperparameters
selected_hyperparameters_df.to_csv(
    tuning_results_dir / "selected_hyperparameters.csv",
    index=False
)

print("\nHyperparameter tuning completed.")
print("Full tuning results saved to:", tuning_results_dir / "hyperparameter_tuning_results.csv")
print("Selected hyperparameters saved to:", tuning_results_dir / "selected_hyperparameters.csv")

print("\nFull tuning results:")
display(tuning_results_df)

print("\nSelected hyperparameters:")
display(selected_hyperparameters_df)

Project folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project
Dataset folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\dataset
Tuning results folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\Data_Scientist\results

Loading dataset for hyperparameter tuning...
Found 4060 files belonging to 5 classes.
Found 820 files belonging to 5 classes.
Dataset loaded successfully.
Class names: ['leaf_blight', 'leaf_rust', 'leaf_spot', 'powdery_mildew', 'yellow_leaf_disease']
Number of classes: 5

Starting tuning for MobileNetV3Small

MobileNetV3Small - Trial 1
Learning rate: 0.001
Dropout: 0.3
Trainable layers: Frozen base
Epoch 1/3
127/127 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - accuracy: 0.3719 - loss: 1.5121 - val_accuracy: 0.5463 - val_loss: 1.1986
Epoch 2/3
127/127 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.5071 - loss: 1.2363 - val_accuracy: 0.5854 - val_loss: 1.1197
Epoch 3/3
127/127 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.5569 - loss: 1.1364 - v

,Model,Trial,Learning Rate,Dropout,Trainable Layers,Epochs,Best Validation Accuracy,Best Validation Accuracy (%),Final Validation Accuracy,Final Validation Accuracy (%),Final Validation Loss,Training Time (seconds),Training Time (minutes),Total Parameters
0,MobileNetV3Small,Trial 1,0.00100,0.3,Frozen base,3,0.603659,60.37,0.603659,60.37,1.0712,31.16,0.52,942005
1,MobileNetV3Small,Trial 2,0.00010,0.4,Frozen base,3,0.352439,35.24,0.352439,35.24,1.4969,31.24,0.52,942005
2,MobileNetV3Small,Trial 3,0.00001,0.3,Fine-tune last 10 layers,3,0.278049,27.80,0.278049,27.80,1.6144,32.63,0.54,942005
3,ResNet50,Trial 1,0.00100,0.3,Frozen base,3,0.631707,63.17,0.631707,63.17,0.9788,344.97,5.75,23597957
4,ResNet50,Trial 2,0.00010,0.4,Frozen base,3,0.484146,48.41,0.484146,48.41,1.2819,373.91,6.23,23597957
5,ResNet50,Trial 3,0.00001,0.3,Fine-tune last 10 layers,3,0.489024,48.90,0.489024,48.90,1.2725,416.24,6.94,23597957
6,DenseNet121,Trial 1,0.00100,0.3,Frozen base,3,0.582927,58.29,0.582927,58.29,1.0897,406.20,6.77,7042629
7,DenseNet121,Trial 2,0.00010,0.4,Frozen base,3,0.351220,35.12,0.351220,35.12,1.5268,396.54,6.61,7042629
8,DenseNet121,Trial 3,0.00001,0.3,Fine-tune last 10 layers,3,0.269512,26.95,0.269512,26.95,1.6343,412.36,6.87,7042629



Selected hyperparameters:


,Model,Trial,Learning Rate,Dropout,Trainable Layers,Epochs,Best Validation Accuracy (%),Final Validation Accuracy (%),Training Time (minutes)
0,DenseNet121,Trial 1,0.001,0.3,Frozen base,3,58.29,58.29,6.77
1,MobileNetV3Small,Trial 1,0.001,0.3,Frozen base,3,60.37,60.37,0.52
2,ResNet50,Trial 1,0.001,0.3,Frozen base,3,63.17,63.17,5.75


## 8. MobileNetV3Small Training and Evaluation

This section builds MobileNetV3Small using transfer learning, trains it for 50 epochs, evaluates it on the testing dataset, calculates mAP, and saves the result.

In [8]:
import time
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Small

# Create a fresh MobileNetV3Small model for final training
mobilenet_base = MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

# Freeze pre-trained layers
mobilenet_base.trainable = False

# Build final classification model
mobilenet_model = models.Sequential([
    mobilenet_base,
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

# Compile model
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Show model details
mobilenet_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)        │ (None, 576)                 │         939,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 576)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │           2,885 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 942,005 (3.59 MB)

 Trainable params: 2,885 (11.27 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [9]:
# Train MobileNetV3Small for 50 epochs
start_time = time.time()

mobilenet_history = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

end_time = time.time()

mobilenet_training_time = end_time - start_time

print("MobileNetV3Small training completed.")
print("Training time:", round(mobilenet_training_time, 2), "seconds")
print("Training time:", round(mobilenet_training_time / 60, 2), "minutes")

Epoch 1/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - accuracy: 0.3451 - loss: 1.5552 - val_accuracy: 0.5207 - val_loss: 1.2251
Epoch 2/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.5113 - loss: 1.2387 - val_accuracy: 0.5768 - val_loss: 1.1184
Epoch 3/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.5574 - loss: 1.1384 - val_accuracy: 0.5866 - val_loss: 1.0880
Epoch 4/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.5840 - loss: 1.0911 - val_accuracy: 0.5951 - val_loss: 1.0579
Epoch 5/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.6079 - loss: 1.0404 - val_accuracy: 0.5963 - val_loss: 1.0436
Epoch 6/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.6076 - loss: 1.0281 - val_accuracy: 0.6061 - val_loss: 1.0391
Epoch 7/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.6222 - loss: 1.0042 - val_accuracy: 0.6049 - val_loss: 1.0312
Epoch 8/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 9s 68ms/step - accuracy: 0.6133 - loss: 1.0073 - val_acc

In [10]:
# Evaluate MobileNetV3Small using the test dataset
mobilenet_test_loss, mobilenet_test_accuracy = mobilenet_model.evaluate(test_ds)

print("MobileNetV3Small Test Loss:", round(mobilenet_test_loss, 4))
print("MobileNetV3Small Test Accuracy:", round(mobilenet_test_accuracy, 4))
print("MobileNetV3Small Test Accuracy (%):", round(mobilenet_test_accuracy * 100, 2), "%")

28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.6264 - loss: 0.9785
MobileNetV3Small Test Loss: 0.9785
MobileNetV3Small Test Accuracy: 0.6264
MobileNetV3Small Test Accuracy (%): 62.64 %


In [11]:
import numpy as np
import tensorflow as tf

# Try importing average_precision_score
try:
    from sklearn.metrics import average_precision_score
    print("scikit-learn is available.")
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install scikit-learn
    from sklearn.metrics import average_precision_score
    print("scikit-learn installed and imported.")

# Get true labels and predicted probabilities
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    predictions = mobilenet_model.predict(images, verbose=0)
    y_pred_probs.extend(predictions)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

# Convert true labels to one-hot format
y_true_one_hot = tf.keras.utils.to_categorical(y_true, num_classes=num_classes)

# Calculate mAP
mobilenet_map = average_precision_score(
    y_true_one_hot,
    y_pred_probs,
    average="macro"
)

print("MobileNetV3Small mAP:", round(mobilenet_map, 4))
print("MobileNetV3Small mAP (%):", round(mobilenet_map * 100, 2), "%")

scikit-learn is available.
MobileNetV3Small mAP: 0.6899
MobileNetV3Small mAP (%): 68.99 %


In [12]:
import pandas as pd

# Count model parameters
mobilenet_params = mobilenet_model.count_params()

# Create result table for MobileNetV3Small
mobilenet_result = {
    "Model": "MobileNetV3Small",
    "Test Accuracy": mobilenet_test_accuracy,
    "Test Accuracy (%)": mobilenet_test_accuracy * 100,
    "Test Loss": mobilenet_test_loss,
    "mAP": mobilenet_map,
    "mAP (%)": mobilenet_map * 100,
    "Training Time (seconds)": mobilenet_training_time,
    "Training Time (minutes)": mobilenet_training_time / 60,
    "Total Parameters": mobilenet_params
}

mobilenet_result_df = pd.DataFrame([mobilenet_result])

# Save result table
mobilenet_result_df.to_csv(results_dir / "mobilenetv3small_result.csv", index=False)

# Save training history
mobilenet_history_df = pd.DataFrame(mobilenet_history.history)
mobilenet_history_df.to_csv(results_dir / "mobilenetv3small_history.csv", index=False)

# Save trained model
mobilenet_model.save(models_dir / "mobilenetv3small_model.keras")

print("MobileNetV3Small result saved successfully.")
print("Result file:", results_dir / "mobilenetv3small_result.csv")
print("History file:", results_dir / "mobilenetv3small_history.csv")
print("Model file:", models_dir / "mobilenetv3small_model.keras")

mobilenet_result_df

MobileNetV3Small result saved successfully.
Result file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\mobilenetv3small_result.csv
History file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\mobilenetv3small_history.csv
Model file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\saved_models\mobilenetv3small_model.keras


,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
0,MobileNetV3Small,0.626437,62.643677,0.978535,0.68987,68.987027,492.150125,8.202502,942005


## 9. ResNet50 Training and Evaluation

This section builds ResNet50 using transfer learning, trains it for 50 epochs, evaluates it on the testing dataset, calculates mAP, and saves the result.

In [16]:
import time
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

# Create fresh ResNet50 base model
resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

# Freeze pre-trained layers
resnet_base.trainable = False

# Build fresh ResNet50 model
inputs = tf.keras.Input(shape=(224, 224, 3))

x = layers.Lambda(resnet_preprocess)(inputs)
x = resnet_base(x, training=False)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)

resnet_model = tf.keras.Model(inputs, outputs, name="ResNet50_Model")

# Compile model
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

resnet_model.summary()

Model: "ResNet50_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda_2 (Lambda)                    │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ resnet50 (Functional)                │ (None, 2048)                │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 5)                   │          10,245 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [17]:
# Train ResNet50 for 50 epochs
start_time = time.time()

resnet_history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

end_time = time.time()

resnet_training_time = end_time - start_time

print("ResNet50 training completed.")
print("Training time:", round(resnet_training_time, 2), "seconds")
print("Training time:", round(resnet_training_time / 60, 2), "minutes")

Epoch 1/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 106s 806ms/step - accuracy: 0.4374 - loss: 1.4715 - val_accuracy: 0.5610 - val_loss: 1.1655
Epoch 2/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 109s 859ms/step - accuracy: 0.5975 - loss: 1.0788 - val_accuracy: 0.6146 - val_loss: 1.0486
Epoch 3/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 937ms/step - accuracy: 0.6416 - loss: 0.9594 - val_accuracy: 0.6500 - val_loss: 0.9641
Epoch 4/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 118s 927ms/step - accuracy: 0.6663 - loss: 0.8876 - val_accuracy: 0.6305 - val_loss: 0.9837
Epoch 5/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 120s 944ms/step - accuracy: 0.6833 - loss: 0.8435 - val_accuracy: 0.6622 - val_loss: 0.9588
Epoch 6/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 118s 933ms/step - accuracy: 0.6990 - loss: 0.8048 - val_accuracy: 0.6683 - val_loss: 0.9119
Epoch 7/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 936ms/step - accuracy: 0.7190 - loss: 0.7720 - val_accuracy: 0.6720 - val_loss: 0.9100
Epoch 8/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 936ms/step - accuracy: 0.7217 -

In [18]:
# Evaluate ResNet50 using the test dataset
resnet_test_loss, resnet_test_accuracy = resnet_model.evaluate(test_ds)

print("ResNet50 Test Loss:", round(resnet_test_loss, 4))
print("ResNet50 Test Accuracy:", round(resnet_test_accuracy, 4))
print("ResNet50 Test Accuracy (%):", round(resnet_test_accuracy * 100, 2), "%")

28/28 ━━━━━━━━━━━━━━━━━━━━ 17s 616ms/step - accuracy: 0.7115 - loss: 0.8988
ResNet50 Test Loss: 0.8988
ResNet50 Test Accuracy: 0.7115
ResNet50 Test Accuracy (%): 71.15 %


In [19]:
import numpy as np
import tensorflow as tf

try:
    from sklearn.metrics import average_precision_score
    print("scikit-learn is available.")
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install scikit-learn
    from sklearn.metrics import average_precision_score
    print("scikit-learn installed and imported.")

# Get true labels and predicted probabilities
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    predictions = resnet_model.predict(images, verbose=0)
    y_pred_probs.extend(predictions)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

# Convert true labels to one-hot format
y_true_one_hot = tf.keras.utils.to_categorical(y_true, num_classes=num_classes)

# Calculate mAP
resnet_map = average_precision_score(
    y_true_one_hot,
    y_pred_probs,
    average="macro"
)

print("ResNet50 mAP:", round(resnet_map, 4))
print("ResNet50 mAP (%):", round(resnet_map * 100, 2), "%")

scikit-learn is available.
ResNet50 mAP: 0.7856
ResNet50 mAP (%): 78.56 %


In [20]:
import pandas as pd

# Count model parameters
resnet_params = resnet_model.count_params()

# Create result table for ResNet50
resnet_result = {
    "Model": "ResNet50",
    "Test Accuracy": resnet_test_accuracy,
    "Test Accuracy (%)": resnet_test_accuracy * 100,
    "Test Loss": resnet_test_loss,
    "mAP": resnet_map,
    "mAP (%)": resnet_map * 100,
    "Training Time (seconds)": resnet_training_time,
    "Training Time (minutes)": resnet_training_time / 60,
    "Total Parameters": resnet_params
}

resnet_result_df = pd.DataFrame([resnet_result])

# Save result table
resnet_result_df.to_csv(results_dir / "resnet50_result.csv", index=False)

# Save training history
resnet_history_df = pd.DataFrame(resnet_history.history)
resnet_history_df.to_csv(results_dir / "resnet50_history.csv", index=False)

# Save trained model
resnet_model.save(models_dir / "resnet50_model.keras")

print("ResNet50 result saved successfully.")
print("Result file:", results_dir / "resnet50_result.csv")
print("History file:", results_dir / "resnet50_history.csv")
print("Model file:", models_dir / "resnet50_model.keras")

resnet_result_df

ResNet50 result saved successfully.
Result file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\resnet50_result.csv
History file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\resnet50_history.csv
Model file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\saved_models\resnet50_model.keras


,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
0,ResNet50,0.711494,71.149427,0.898819,0.785578,78.557766,5880.304603,98.005077,23597957


## 10. Memory Cleanup

This section clears previous large models from memory before training DenseNet121.

In [21]:
import gc
import tensorflow as tf

# Remove previous large models from memory
try:
    del resnet_model
    del resnet_base
except:
    pass

try:
    del mobilenet_model
    del mobilenet_base
except:
    pass

tf.keras.backend.clear_session()
gc.collect()

print("Memory cleaned. Ready for DenseNet121.")

Memory cleaned. Ready for DenseNet121.


## 11. DenseNet121 Training and Evaluation

This section builds DenseNet121 using transfer learning, trains it for 50 epochs, evaluates it on the testing dataset, calculates mAP, and saves the result.

In [24]:
import time
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

# Create fresh DenseNet121 base model
densenet_base = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

# Freeze pre-trained layers
densenet_base.trainable = False

# Build fresh DenseNet121 model
inputs = tf.keras.Input(shape=(224, 224, 3))

x = layers.Lambda(densenet_preprocess)(inputs)
x = densenet_base(x, training=False)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)

densenet_model = tf.keras.Model(inputs, outputs, name="DenseNet121_Model")

# Compile model
densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

densenet_model.summary()

Model: "DenseNet121_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda_1 (Lambda)                    │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ densenet121 (Functional)             │ (None, 1024)                │       7,037,504 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │           5,125 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,042,629 (26.87 MB)

 Trainable params: 5,125 (20.02 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [25]:
# Train DenseNet121 for 50 epochs
start_time = time.time()

densenet_history = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

end_time = time.time()

densenet_training_time = end_time - start_time

print("DenseNet121 training completed.")
print("Training time:", round(densenet_training_time, 2), "seconds")
print("Training time:", round(densenet_training_time / 60, 2), "minutes")

Epoch 1/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 110s 818ms/step - accuracy: 0.3552 - loss: 1.6181 - val_accuracy: 0.4927 - val_loss: 1.2407
Epoch 2/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 112s 882ms/step - accuracy: 0.4862 - loss: 1.2856 - val_accuracy: 0.5561 - val_loss: 1.1240
Epoch 3/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 136s 836ms/step - accuracy: 0.5515 - loss: 1.1483 - val_accuracy: 0.5756 - val_loss: 1.0768
Epoch 4/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 940ms/step - accuracy: 0.5680 - loss: 1.1154 - val_accuracy: 0.6134 - val_loss: 1.0509
Epoch 5/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 118s 931ms/step - accuracy: 0.5951 - loss: 1.0761 - val_accuracy: 0.6037 - val_loss: 1.0374
Epoch 6/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 938ms/step - accuracy: 0.5978 - loss: 1.0525 - val_accuracy: 0.6098 - val_loss: 1.0269
Epoch 7/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 938ms/step - accuracy: 0.6047 - loss: 1.0250 - val_accuracy: 0.5951 - val_loss: 1.0288
Epoch 8/50
127/127 ━━━━━━━━━━━━━━━━━━━━ 119s 934ms/step - accuracy: 0.6116 -

In [26]:
# Evaluate DenseNet121 using the test dataset
densenet_test_loss, densenet_test_accuracy = densenet_model.evaluate(test_ds)

print("DenseNet121 Test Loss:", round(densenet_test_loss, 4))
print("DenseNet121 Test Accuracy:", round(densenet_test_accuracy, 4))
print("DenseNet121 Test Accuracy (%):", round(densenet_test_accuracy * 100, 2), "%")

28/28 ━━━━━━━━━━━━━━━━━━━━ 18s 638ms/step - accuracy: 0.6356 - loss: 0.9478
DenseNet121 Test Loss: 0.9478
DenseNet121 Test Accuracy: 0.6356
DenseNet121 Test Accuracy (%): 63.56 %


In [27]:
import numpy as np
import tensorflow as tf

try:
    from sklearn.metrics import average_precision_score
    print("scikit-learn is available.")
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install scikit-learn
    from sklearn.metrics import average_precision_score
    print("scikit-learn installed and imported.")

# Get true labels and predicted probabilities
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    predictions = densenet_model.predict(images, verbose=0)
    y_pred_probs.extend(predictions)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

# Convert true labels to one-hot format
y_true_one_hot = tf.keras.utils.to_categorical(y_true, num_classes=num_classes)

# Calculate mAP
densenet_map = average_precision_score(
    y_true_one_hot,
    y_pred_probs,
    average="macro"
)

print("DenseNet121 mAP:", round(densenet_map, 4))
print("DenseNet121 mAP (%):", round(densenet_map * 100, 2), "%")

scikit-learn is available.
DenseNet121 mAP: 0.7058
DenseNet121 mAP (%): 70.58 %


In [28]:
import pandas as pd

# Count model parameters
densenet_params = densenet_model.count_params()

# Create result table for DenseNet121
densenet_result = {
    "Model": "DenseNet121",
    "Test Accuracy": densenet_test_accuracy,
    "Test Accuracy (%)": densenet_test_accuracy * 100,
    "Test Loss": densenet_test_loss,
    "mAP": densenet_map,
    "mAP (%)": densenet_map * 100,
    "Training Time (seconds)": densenet_training_time,
    "Training Time (minutes)": densenet_training_time / 60,
    "Total Parameters": densenet_params
}

densenet_result_df = pd.DataFrame([densenet_result])

# Save result table
densenet_result_df.to_csv(results_dir / "densenet121_result.csv", index=False)

# Save training history
densenet_history_df = pd.DataFrame(densenet_history.history)
densenet_history_df.to_csv(results_dir / "densenet121_history.csv", index=False)

# Save trained model
densenet_model.save(models_dir / "densenet121_model.keras")

print("DenseNet121 result saved successfully.")
print("Result file:", results_dir / "densenet121_result.csv")
print("History file:", results_dir / "densenet121_history.csv")
print("Model file:", models_dir / "densenet121_model.keras")

densenet_result_df

DenseNet121 result saved successfully.
Result file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\densenet121_result.csv
History file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\densenet121_history.csv
Model file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\saved_models\densenet121_model.keras


,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
0,DenseNet121,0.635632,63.563216,0.947757,0.705842,70.584228,5988.468668,99.807811,7042629


## 12. Final Model Comparison

This section combines the result files from all three models into one final comparison table. The models are compared using test accuracy, mAP, training time, and total parameters.

In [29]:
# Load all saved result CSV files
mobilenet_result_df = pd.read_csv(results_dir / "mobilenetv3small_result.csv")
resnet_result_df = pd.read_csv(results_dir / "resnet50_result.csv")
densenet_result_df = pd.read_csv(results_dir / "densenet121_result.csv")

# Combine into one final comparison table
final_results_df = pd.concat(
    [mobilenet_result_df, resnet_result_df, densenet_result_df],
    ignore_index=True
)

# Sort by test accuracy, highest first
final_results_df = final_results_df.sort_values(
    by="Test Accuracy (%)",
    ascending=False
)

# Save final comparison table
final_results_df.to_csv(results_dir / "final_model_comparison.csv", index=False)

print("Final model comparison saved successfully.")
print("File:", results_dir / "final_model_comparison.csv")

final_results_df

Final model comparison saved successfully.
File: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\final_model_comparison.csv


,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
1,ResNet50,0.711494,71.149427,0.898819,0.785578,78.557766,5880.304603,98.005077,23597957
2,DenseNet121,0.635632,63.563216,0.947757,0.705842,70.584228,5988.468668,99.807811,7042629
0,MobileNetV3Small,0.626437,62.643677,0.978535,0.689870,68.987027,492.150125,8.202502,942005


In [30]:
# Round final comparison table numbers for cleaner display

rounded_results_df = final_results_df.copy()

# Round percentage columns to 2 decimal places
rounded_results_df["Test Accuracy (%)"] = rounded_results_df["Test Accuracy (%)"].round(2)
rounded_results_df["mAP (%)"] = rounded_results_df["mAP (%)"].round(2)
rounded_results_df["Training Time (minutes)"] = rounded_results_df["Training Time (minutes)"].round(2)

# Round decimal columns to 4 decimal places
rounded_results_df["Test Accuracy"] = rounded_results_df["Test Accuracy"].round(4)
rounded_results_df["Test Loss"] = rounded_results_df["Test Loss"].round(4)
rounded_results_df["mAP"] = rounded_results_df["mAP"].round(4)

# Round seconds to 2 decimal places
rounded_results_df["Training Time (seconds)"] = rounded_results_df["Training Time (seconds)"].round(2)

# Save the rounded table
rounded_results_df.to_csv(results_dir / "final_model_comparison_rounded.csv", index=False)

print("Rounded final model comparison saved successfully.")
print("File:", results_dir / "final_model_comparison_rounded.csv")

rounded_results_df

Rounded final model comparison saved successfully.
File: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\results\final_model_comparison_rounded.csv


,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
1,ResNet50,0.7115,71.15,0.8988,0.7856,78.56,5880.30,98.01,23597957
2,DenseNet121,0.6356,63.56,0.9478,0.7058,70.58,5988.47,99.81,7042629
0,MobileNetV3Small,0.6264,62.64,0.9785,0.6899,68.99,492.15,8.20,942005


In [31]:
rounded_results_df = rounded_results_df.reset_index(drop=True)

rounded_results_df.to_csv(results_dir / "final_model_comparison_rounded.csv", index=False)

rounded_results_df

,Model,Test Accuracy,Test Accuracy (%),Test Loss,mAP,mAP (%),Training Time (seconds),Training Time (minutes),Total Parameters
0,ResNet50,0.7115,71.15,0.8988,0.7856,78.56,5880.30,98.01,23597957
1,DenseNet121,0.6356,63.56,0.9478,0.7058,70.58,5988.47,99.81,7042629
2,MobileNetV3Small,0.6264,62.64,0.9785,0.6899,68.99,492.15,8.20,942005


## 13. Saved Models

This section compresses the saved trained models into a ZIP file. The ZIP file is uploaded to Google Drive because the model files are too large for GitHub.

In [1]:
from pathlib import Path
import shutil

project_dir = Path.cwd()

# If currently inside Data_Scientist folder, go back to Project folder
if project_dir.name == "Data_Scientist":
    project_dir = project_dir.parent

saved_models_dir = project_dir / "saved_models"
output_zip = project_dir / "saved_models"

if saved_models_dir.exists():
    zip_path = shutil.make_archive(
        base_name=str(output_zip),
        format="zip",
        root_dir=saved_models_dir
    )
    
    print("Saved models zipped successfully.")
    print("ZIP file:", zip_path)
else:
    print("saved_models folder not found.")

Saved models zipped successfully.
ZIP file: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\saved_models.zip


## 14. Prediction Labels for Confusion Matrix

This section generates prediction CSV files for the Data Analyst. Each CSV file contains the actual label, actual class, predicted label, predicted class, and prediction probabilities. These files are used to create the confusion matrix.

In [1]:
from pathlib import Path
import tensorflow as tf
import pandas as pd
import numpy as np

from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

# Locate project folder
project_dir = Path.cwd()

# If notebook is inside Data_Scientist folder, go back to Project folder
if project_dir.name == "Data_Scientist":
    project_dir = project_dir.parent

dataset_dir = project_dir / "dataset"
saved_models_dir = project_dir / "saved_models"
results_dir = project_dir / "Data_Scientist" / "results"

results_dir.mkdir(exist_ok=True)

print("Project folder:", project_dir)
print("Dataset folder:", dataset_dir)
print("Saved models folder:", saved_models_dir)
print("Results folder:", results_dir)

Project folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project
Dataset folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\dataset
Saved models folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\saved_models
Results folder: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\Data_Scientist\results


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

class_names = test_ds.class_names
num_classes = len(class_names)

print("Class names:", class_names)
print("Number of classes:", num_classes)

Found 870 files belonging to 5 classes.
Class names: ['leaf_blight', 'leaf_rust', 'leaf_spot', 'powdery_mildew', 'yellow_leaf_disease']
Number of classes: 5


In [7]:
def generate_prediction_csv(model_name, model_path, output_file, custom_objects=None):
    print("\nLoading:", model_name)
    
    model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        safe_mode=False,
        compile=False
    )
    
    print("Predicting:", model_name)
    
    # Get prediction probabilities
    y_pred_probs = model.predict(test_ds)
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    
    # Get actual labels
    y_true_labels = []
    
    for images, labels in test_ds:
        y_true_labels.extend(labels.numpy())
    
    y_true_labels = np.array(y_true_labels)
    
    # Convert labels to class names
    y_true_classes = [class_names[label] for label in y_true_labels]
    y_pred_classes = [class_names[label] for label in y_pred_labels]
    
    # Create prediction dataframe
    prediction_df = pd.DataFrame({
        "Actual Label": y_true_labels,
        "Actual Class": y_true_classes,
        "Predicted Label": y_pred_labels,
        "Predicted Class": y_pred_classes
    })
    
    # Add probability for each class
    for i, class_name in enumerate(class_names):
        prediction_df[f"Probability_{class_name}"] = y_pred_probs[:, i]
    
    # Save CSV
    prediction_df.to_csv(output_file, index=False)
    
    print("Saved:", output_file)
    print(prediction_df.head())
    
    return prediction_df

In [8]:
mobilenet_predictions = generate_prediction_csv(
    model_name="MobileNetV3Small",
    model_path=saved_models_dir / "mobilenetv3small_model.keras",
    output_file=results_dir / "mobilenetv3small_predictions.csv"
)

resnet_predictions = generate_prediction_csv(
    model_name="ResNet50",
    model_path=saved_models_dir / "resnet50_model.keras",
    output_file=results_dir / "resnet50_predictions.csv",
    custom_objects={
        "preprocess_input": resnet_preprocess
    }
)

densenet_predictions = generate_prediction_csv(
    model_name="DenseNet121",
    model_path=saved_models_dir / "densenet121_model.keras",
    output_file=results_dir / "densenet121_predictions.csv",
    custom_objects={
        "preprocess_input": densenet_preprocess
    }
)

print("All prediction CSV files created successfully.")


Loading: MobileNetV3Small
Predicting: MobileNetV3Small
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step
Saved: C:\Users\ASUS\anaconda3\envs\ISB46703\Lab work\Project\Data_Scientist\results\mobilenetv3small_predictions.csv
   Actual Label Actual Class  Predicted Label      Predicted Class  \
0             0  leaf_blight                1            leaf_rust   
1             0  leaf_blight                2            leaf_spot   
2             0  leaf_blight                3       powdery_mildew   
3             0  leaf_blight                4  yellow_leaf_disease   
4             0  leaf_blight                3       powdery_mildew   

   Probability_leaf_blight  Probability_leaf_rust  Probability_leaf_spot  \
0                 0.028899               0.480905               0.079243   
1                 0.123476               0.113733               0.447269   
2                 0.126776               0.066644               0.233455   
3                 0.116149               0.222625            

In [9]:
for file_name in [
    "mobilenetv3small_predictions.csv",
    "resnet50_predictions.csv",
    "densenet121_predictions.csv"
]:
    file_path = results_dir / file_name
    print(file_name, "exists:", file_path.exists())

mobilenetv3small_predictions.csv exists: True
resnet50_predictions.csv exists: True
densenet121_predictions.csv exists: True


## 15. Conclusion

Based on the final model comparison, ResNet50 achieved the best classification performance with the highest test accuracy and mAP. MobileNetV3Small was the fastest and lightest model, but its accuracy was lower compared to ResNet50.

Therefore, ResNet50 is selected as the best model for plant leaf disease classification performance, while MobileNetV3Small is more suitable for lightweight applications.